# The Experiment-Planning Playbook
### Every design the framework supports, how to power it, price it, pick it — and fold it back into the MMM

An MMM tells you where the model is *uncertain*; an experiment is how you buy the
missing information. This notebook demonstrates the full planning surface on one
synthetic world with a **known answer key**:

1. **The method registry** — SCM, TBR/CausalImpact, GBR, DiD-MMT, ghost ads,
   switchbacks, flighting — one catalogue, filtered by what your data supports.
2. **Planning each design** with honest power math (matched pairs, ITT dilution,
   carryover-aware blocks, AR(1) design effects) — including power on the
   client's scale: **maximum detectable cost per conversion**, and why the
   skewed CPA distribution can't be produced by flipping the lift numbers.
3. **The methodology leaderboard** — audition every estimator on your own history
   (A/A false-positive rate before A/B power).
4. **Which test to run** — the EIG/EVOI priority grid, including **non-geo**
   designs through the `sigma_exp` bridge.
5. **What it's worth** — opportunity cost, net test value, and the **net-value
   Pareto optimizer** (new: a two-anchor Gaussian EVOI surrogate prices every
   candidate design in dollars).
6. **Closing the loop** — the readout becomes a calibration likelihood in the
   next fit.

Companion doc: [`docs/experiment-playbook.html`](../../docs/experiment-playbook.html).

In [14]:
import warnings; warnings.filterwarnings("ignore")
import os
os.environ.setdefault("TQDM_DISABLE", "1")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.templates.default = "plotly_white"
pio.renderers.default = "notebook_connected"
pd.set_option("display.width", 160)

import logging
from loguru import logger
logger.disable("mmm_framework")
for _n in ("pymc", "pymc.sampling", "numpyro", "jax", "arviz"):
    logging.getLogger(_n).setLevel(logging.ERROR)

# palette (matches the docs site's editorial scheme)
INK, MUTED = "#1f2430", "#8a8f98"
GOOD, BAD, GOLD = "#3d7a5c", "#b4552d", "#c9962e"
PALETTE = {"TV": "#4464ad", "Search": "#c9962e", "Social": "#3d7a5c", "Display": "#b4552d"}

def style(fig, height=380, title=None, **kw):
    fig.update_layout(height=height, title=title, margin=dict(t=60, l=60, r=30, b=50),
                      font=dict(size=12), **kw)
    return fig

KPI = "Sales"
print("Experiment-planning playbook — setup ready.")

Experiment-planning playbook — setup ready.


## The world — a geo panel with a known answer key

Eight markets, a year and a half of weekly data, four channels. Because the
world is synthetic we know the true ROAS of every channel — the luxury no real
dataset gives you, and the reason a playbook should be rehearsed on synthetic
ground truth first. (A *deliberately* modest history: with only 78 weeks the
posterior stays wide enough that experiments have real value to price — a
longer, cleaner panel would leave less for a test to teach.) We also keep a
**national** spine (no geo splits) to demonstrate the designs you fall back on
when there is no panel.

In [15]:
import tempfile
from mmm_framework.synth import dgp_geo, generate_mff
from mmm_framework.synth.mff import geo_scenario_to_mff

WORK = os.path.join(tempfile.gettempdir(), "mmm_experiment_playbook")
os.makedirs(WORK, exist_ok=True)

GEOS = ["North", "South", "East", "West", "Metro", "Coast", "Plains", "Hills"]
sc = dgp_geo.build("geo_heterogeneous", seed=3, geos=GEOS, n_weeks=78)
geo_df = geo_scenario_to_mff(sc)
GEO_CSV = os.path.join(WORK, "geo_world.csv")
geo_df.to_csv(GEO_CSV, index=False)

nat_df, nat_key = generate_mff("realistic", seed=5, n_weeks=130)
NAT_CSV = os.path.join(WORK, "national_world.csv")
nat_df.to_csv(NAT_CSV, index=False)

CHANNELS = ["TV", "Search", "Social", "Display"]
print(f"geo world:      {len(GEOS)} geos x 78 weeks -> {GEO_CSV}")
print(f"national world: 1 geo x 130 weeks           -> {NAT_CSV}")

geo world:      8 geos x 78 weeks -> /var/folders/nw/92nd031j7p5d2bs8grpysp4w0000gn/T/mmm_experiment_playbook/geo_world.csv
national world: 1 geo x 130 weeks           -> /var/folders/nw/92nd031j7p5d2bs8grpysp4w0000gn/T/mmm_experiment_playbook/national_world.csv


## 1 · The method registry — one catalogue of experiment methodologies

Every supported methodology is a named `MethodSpec` in `planning/methods/`:
its data requirements, its estimator, its power math, its references. The
design tools, the REST API and the agent all enumerate this same registry —
"which methods can my data support?" has one answer everywhere.

In [16]:
from mmm_framework.planning.methods import list_methods, methods_for_data

cat = pd.DataFrame([{
    "key": m.key, "name": m.name, "family": m.requirement.family,
    "min_geos": m.requirement.min_geos or "-",
    "min_pre_weeks": m.requirement.min_pre_weeks,
    "needs_panel": m.requirement.needs_panel,
} for m in list_methods()])
print("the registry:")
print(cat.to_string(index=False))

# ...filtered by what THIS dataset can actually support:
support = pd.DataFrame(methods_for_data(n_geos=len(GEOS), n_weeks=78))
print("\nsupported on the 8-geo panel:")
print(support[["key", "supported", "reason"]].fillna("").to_string(index=False))

the registry:
              key                                       name     family min_geos  min_pre_weeks  needs_panel
          did_mmt               Matched-market DiD (DiD-MMT)        geo        4              8         True
              gbr                 Geo-based regression (GBR)        geo        4              8         True
       regadj_geo            Regression-adjusted geo (ridge)        geo        4             12         True
synthetic_control                          Synthetic control        geo        4             12         True
              tbr Time-based regression (TBR / CausalImpact)        geo        2             12         True
       switchback               Switchback (time-randomized) switchback        -              8        False
        ghost_ads                 Ghost ads (user-level RCT)       user        -              8        False

supported on the 8-geo panel:
              key  supported                             reason
          did_mmt  

## 2 · Planning a geo experiment — matched pairs + a power curve

`design_options` says what the data supports; `design_experiment` builds a
runnable plan: markets matched into pairs on pre-period behaviour, treatment
randomized within pair, and a **power curve** — the minimum detectable effect
(MDE) at every candidate duration. The MDE is the design's blind spot: an
effect smaller than it will come back "inconclusive" no matter what is true.

In [17]:
from mmm_framework.planning import design_experiment, design_options

opts = design_options(GEO_CSV, KPI, "TV")
print(f"design families supported: {opts['designs']}   recommended: {opts['recommended']}")

plan = design_experiment(GEO_CSV, KPI, "TV", design="holdout", n_pairs=4, duration=8)
asg = pd.DataFrame(plan["assignment"])[["treatment", "control", "correlation", "residual_correlation"]]
print("\nmatched pairs (treated market goes dark; its twin keeps spending):")
print(asg.round(3).to_string(index=False))
print(f"\nMDE(ROAS) at {plan['duration']}w: {plan['mde_roas']:.2f}   (SE source: {plan['se_source']})")

pc = pd.DataFrame(plan["power_curve"])
fig = go.Figure(go.Scatter(x=pc["duration"], y=pc["mde_roas"], mode="lines+markers",
                           line=dict(color=PALETTE["TV"], width=3), showlegend=False))
fig.add_vline(x=plan["duration"], line=dict(color=MUTED, width=1.5, dash="dash"),
              annotation_text=f"chosen {plan['duration']}w")
style(fig, height=340, title="Geo holdout power curve — longer test, smaller detectable effect")
fig.update_layout(xaxis_title="test duration (weeks)", yaxis_title="MDE (ROAS)")
fig.show()

design families supported: ['geo_lift', 'matched_market_did', 'national_flighting']   recommended: geo_lift

matched pairs (treated market goes dark; its twin keeps spending):
treatment control  correlation  residual_correlation
    Coast    West        0.688                 0.550
     East  Plains        0.595                 0.443
    Metro   Hills        0.595                 0.366
    South   North        0.189                -0.014

MDE(ROAS) at 8w: 0.20   (SE source: placebo_calibrated)


## 3 · Choosing the *analysis* — the same design, five estimators

The footprint and schedule are one decision; **how you read the result out** is
another. Passing `method=` picks the named methodology: the plan carries the
method's metadata and its analysis-plan estimator line is rewritten, so the
pre-registered plan and the eventual readout use the same estimator by
construction. (Where the estimators genuinely differ — validity and power on
*your* history — is the leaderboard's job, next.)

In [18]:
rows = []
for m in ["did_mmt", "synthetic_control", "regadj_geo", "tbr", "gbr"]:
    d = design_experiment(GEO_CSV, KPI, "TV", method=m, design="holdout", n_pairs=4, duration=8)
    rows.append({
        "method": m, "name": d.get("method_name", m),
        "design_key": d["design_key"], "mde_roas": round(d["mde_roas"], 2),
        "references": ", ".join(d.get("method_references") or []) or "-",
    })
print(pd.DataFrame(rows).to_string(index=False))
print("\nSame footprint, same MDE — the method chooses the ESTIMATOR the readout")
print("and the economics simulation will use, not the field design.")

           method                                       name         design_key  mde_roas                         references
          did_mmt               Matched-market DiD (DiD-MMT) matched_market_did      0.16       docs/blog-staggered-did.html
synthetic_control                          Synthetic control           geo_lift      0.20   docs/blog-synthetic-control.html
       regadj_geo            Regression-adjusted geo (ridge)           geo_lift      0.20                                  -
              tbr Time-based regression (TBR / CausalImpact)           geo_lift      0.20 docs/blog-geo-experiments-tbr.html
              gbr                 Geo-based regression (GBR)           geo_lift      0.20                                  -

Same footprint, same MDE — the method chooses the ESTIMATOR the readout
and the economics simulation will use, not the field design.


## 4 · Ghost ads — user-level power with no panel at all

Ghost ads randomize *individuals*: treated users see the real ad, ghost/PSA
users a logged placebo. The randomization lives in the ad server, so this
method needs **no geo panel and no fitted model** — it is a standalone power
calculator. Three honesty devices: **ITT vs TOT** (partial exposure dilutes the
measurable effect), a **rare-event flag** (the normal approximation needs ~30
events per arm), and a **break-even lift** when you price the conversion.

In [19]:
from mmm_framework.planning.methods import (
    GhostAdsDesign, ghost_ads_power, ghost_ads_simulate, ghost_ads_users_for_mde,
)

design = GhostAdsDesign(
    users_reached=400_000, baseline_rate=0.021, treated_fraction=0.5,
    exposure_rate=0.65, value_per_conversion=38.0, cost_per_user=0.05,
)
p = ghost_ads_power(design)
print(f"MDE (absolute lift): {p['mde_abs']:.4f}   relative: {p['mde_rel']:.1%}")
print(f"ITT MDE {p['itt_mde']:.4f}  vs TOT MDE {p['tot_mde']:.4f}  (exposure {p['exposure_rate']:.0%})")
print(f"incremental value at the MDE: ${p['incremental_value_at_mde']:,.0f}  "
      f"media cost: ${p['media_cost']:,.0f}")
print(f"break-even lift: {p['breakeven_lift_abs']:.4f}   rare-event regime: {p['rare_event_regime']}")

need = ghost_ads_users_for_mde(design, target_lift_abs=0.002)
print(f"\nusers required for a +0.2pp MDE: {need:,}")

sim = ghost_ads_simulate(design, p["mde_abs"])
print(f"Monte-Carlo check — empirical FPR: {sim['empirical_fpr']:.1%} (should be ~5%), "
      f"empirical power at the MDE: {sim['empirical_power']:.0%} (should be ~80%)")

reach = np.array([50_000, 100_000, 200_000, 400_000, 800_000, 1_600_000])
mdes = [ghost_ads_power(GhostAdsDesign(users_reached=int(n), baseline_rate=0.021,
                                       exposure_rate=0.65))["mde_rel"] for n in reach]
fig = go.Figure(go.Scatter(x=reach, y=mdes, mode="lines+markers",
                           line=dict(color=PALETTE["Search"], width=3), showlegend=False))
style(fig, height=340, title="Ghost ads: reach buys precision (relative MDE vs users reached)")
fig.update_layout(xaxis_title="users reached", yaxis_title="MDE (relative lift)",
                  xaxis_type="log", yaxis_tickformat=".0%")
fig.show()

MDE (absolute lift): 0.0013   relative: 6.1%
ITT MDE 0.0013  vs TOT MDE 0.0020  (exposure 65%)
incremental value at the MDE: $9,798  media cost: $10,000
break-even lift: 0.0013   rare-event regime: False

users required for a +0.2pp MDE: 168,869
Monte-Carlo check — empirical FPR: 5.1% (should be ~5%), empirical power at the MDE: 80% (should be ~80%)


### Cost per conversion — the number clients ask for, and why you can't just flip the lift

Clients rarely ask "what lift can this test detect?". They ask **"what did a
conversion cost me?"** — spend divided by the extra conversions the media
caused. It is tempting to take the lift analysis and simply divide: flip the
estimate, flip the error bars, done. **That flip is wrong**, and the reason is
worth one minute of intuition:

The *lift* estimate behaves nicely — a symmetric bell curve around the truth.
But cost per conversion **divides a fixed cost by that bell curve**, and
division treats the two sides of the bell very differently. A readout that
comes in a little *high* on conversions moves the cost down a few dollars; a
readout that comes in a little *low* moves the cost up a *lot* — and a readout
near **zero** conversions sends the cost toward infinity. Same symmetric
wobble in, wildly lopsided wobble out.

Three practical consequences:

1. **The average simulated readout overshoots the truth** — the long expensive
   tail drags the mean up, so "expected CPA" quietly flatters to the pessimist
   and the median is the better single number.
2. **The symmetric ± error bar lies.** Flipping the lift interval's width
   (the "delta-method" shortcut) produces an interval that under-covers — and
   in a weak readout it can even dip *below zero* (a negative cost per
   conversion!). The honest interval is **asymmetric**, and when the lift
   interval touches zero it is **unbounded above**: the data genuinely cannot
   rule out an arbitrarily bad CPA, and the report should say so.
3. **The planning number that survives the flip is the *maximum detectable
   cost per conversion*** — invert the detectable-lift *bound*, never the
   point estimate: a design that can detect lift ≥ MDE can certify
   "CPA is at most cost ÷ MDE". Anything worse is indistinguishable from
   "the media did nothing".

`planning.cpa` packages all three (`simulate_cpa_distribution`,
`cpa_interval`, `max_detectable_cpa`, `cpa_power`), and the ghost-ads
calculator now reports its **max detectable CPA** directly. Watch each one on
the 400k-user design.

In [20]:
from mmm_framework.planning import simulate_cpa_distribution

COST_PER_USER = design.cost_per_user
true_lift = 1.25 * p["mde_abs"]          # a realistic, mildly-comfortable truth
sim = simulate_cpa_distribution(true_lift=true_lift, se_lift=p["se_null"],
                                cost=COST_PER_USER, n_sims=20_000, seed=7)
tc = sim["true_cpa"]

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12, subplot_titles=(
    "What the test measures: the LIFT (well-behaved)",
    "What the client sees: the COST PER CONVERSION (skewed)"))
fig.add_trace(go.Histogram(x=sim["lift_draws"] * 1000, nbinsx=60,
                           marker_color=PALETTE["Search"], showlegend=False), row=1, col=1)
fig.add_vline(x=true_lift * 1000, line=dict(color=INK, width=2, dash="dash"),
              annotation_text="truth", row=1, col=1)
show = sim["cpa_draws"][(sim["cpa_draws"] > 0) & (sim["cpa_draws"] < 3 * tc)]
beyond = 100 * (1 - len(show) / len(sim["cpa_draws"]))
fig.add_trace(go.Histogram(x=show, nbinsx=60, marker_color=BAD, showlegend=False),
              row=1, col=2)
fig.add_vline(x=tc, line=dict(color=INK, width=2, dash="dash"),
              annotation_text=f"truth ${tc:.0f}", row=1, col=2)
fig.add_vline(x=sim["median"], line=dict(color=GOOD, width=2),
              annotation_text="median", annotation_position="bottom right", row=1, col=2)
fig.add_vline(x=sim["mean"], line=dict(color=GOLD, width=2),
              annotation_text="mean (dragged up)", row=1, col=2)
fig.add_annotation(x=3 * tc, y=0, xref="x2", yref="paper", ax=-40, ay=-30,
                   text=f"{beyond:.0f}% of readouts land beyond this axis →",
                   font=dict(size=11, color=BAD))
fig.update_xaxes(title_text="measured lift (per 1,000 users)", row=1, col=1)
fig.update_xaxes(title_text="implied cost per conversion ($)", row=1, col=2)
style(fig, height=380,
      title="The SAME 20,000 simulated readouts, before and after the division")
fig.show()

print(f"true CPA: ${tc:.0f}    median readout: ${sim['median']:.0f}    "
      f"mean readout: ${sim['mean']:.0f}  (+{100*(sim['mean_over_true']-1):.0f}% overshoot)")
print(f"skewness of the CPA readout distribution: {sim['skewness']:.0f}  "
      f"(a symmetric bell curve would be ~0)")
print(f"P(readout says MORE THAN DOUBLE the true cost): {sim['p_over_2x_true']:.1%}")
print(f"P(no measurable conversions at all — CPA 'infinite'): {sim['frac_no_positive_lift']:.1%}")

true CPA: $31    median readout: $31    mean readout: $35  (+13% overshoot)
skewness of the CPA readout distribution: 72  (a symmetric bell curve would be ~0)
P(readout says MORE THAN DOUBLE the true cost): 3.7%
P(no measurable conversions at all — CPA 'infinite'): 0.0%


The bell curve survives the test; it does **not** survive the division. And
because the tail only points one way, every summary a client instinctively
reaches for — the mean, the ± error bar — is biased in the *flattering-to-panic*
direction. Here is what that does to the error bars on a single unlucky (but
perfectly plausible) readout, and to how often each interval style actually
contains the truth:

In [21]:
from mmm_framework.planning import cpa_interval

# one comfortable readout and one unlucky-but-plausible one
lucky = cpa_interval(lift=true_lift, se_lift=p["se_null"], cost=COST_PER_USER)
unlucky = cpa_interval(lift=0.6 * p["mde_abs"], se_lift=p["se_null"], cost=COST_PER_USER)

fig = make_subplots(rows=1, cols=2, column_widths=[0.62, 0.38], horizontal_spacing=0.14,
    subplot_titles=("Two interval styles on one unlucky readout",
                    "How often does each interval contain the truth?"))
y_naive, y_honest = 1.0, 0.0
# naive symmetric: dips below zero
fig.add_trace(go.Scatter(x=[unlucky["naive_lo"], unlucky["naive_hi"]], y=[y_naive] * 2,
    mode="lines", line=dict(color=BAD, width=10), opacity=0.5, name="naive ± flip"), row=1, col=1)
fig.add_trace(go.Scatter(x=[unlucky["cpa"]], y=[y_naive], mode="markers",
    marker=dict(color=BAD, size=13), showlegend=False), row=1, col=1)
# honest: lower bound + unbounded above
xmax = max(unlucky["naive_hi"], unlucky["cpa"]) * 1.6
fig.add_trace(go.Scatter(x=[unlucky["lo"], xmax], y=[y_honest] * 2,
    mode="lines", line=dict(color=GOOD, width=10), opacity=0.5,
    name="honest (inverted bound)"), row=1, col=1)
fig.add_trace(go.Scatter(x=[unlucky["cpa"]], y=[y_honest], mode="markers",
    marker=dict(color=GOOD, size=13), showlegend=False), row=1, col=1)
fig.add_annotation(x=xmax, y=y_honest, xref="x1", yref="y1", ax=-8, ay=-26,
    text="unbounded → the data cannot rule out ANY cost", font=dict(size=11, color=GOOD))
fig.add_vline(x=0, line=dict(color=INK, width=1.5, dash="dot"), row=1, col=1)
fig.add_annotation(x=unlucky["naive_lo"], y=y_naive, xref="x1", yref="y1", ax=10, ay=-28,
    text="a NEGATIVE cost per conversion?", font=dict(size=11, color=BAD))
fig.update_yaxes(tickmode="array", tickvals=[y_naive, y_honest],
                 ticktext=["naive ± flip", "honest"], range=[-0.7, 1.7], row=1, col=1)
fig.update_xaxes(title_text="cost per conversion ($)", row=1, col=1)
# coverage
fig.add_trace(go.Bar(x=["naive ± flip", "honest (inverted bound)"],
    y=[sim["coverage_naive"], sim["coverage_inverted"]], marker_color=[BAD, GOOD],
    text=[f"{sim['coverage_naive']:.1%}", f"{sim['coverage_inverted']:.1%}"],
    textposition="outside", showlegend=False), row=1, col=2)
fig.add_hline(y=sim["nominal"], line=dict(color=INK, width=1.5, dash="dot"),
              annotation_text="promised 95%", row=1, col=2)
fig.update_yaxes(tickformat=".0%", range=[0.85, 1.0], row=1, col=2)
style(fig, height=400, title="The symmetric flip lies twice: impossible values AND broken promises",
      legend=dict(orientation="h", y=-0.25))
fig.show()

print(f"unlucky readout (measured lift = 60% of the MDE):")
print(f"  naive ± flip:  ${unlucky['naive_lo']:.0f} .. ${unlucky['naive_hi']:.0f}   "
      "<- a negative lower bound is a nonsense number")
print(f"  honest:        ${unlucky['lo']:.0f} .. unbounded   ({unlucky['status']})")
print(f"comfortable readout: honest interval ${lucky['lo']:.0f} .. ${lucky['hi']:.0f} "
      f"around ${lucky['cpa']:.0f} — note the longer expensive arm ({lucky['status']})")

unlucky readout (measured lift = 60% of the MDE):
  naive ± flip:  $-10 .. $139   <- a negative lower bound is a nonsense number
  honest:        $30 .. unbounded   (upper_unbounded)
comfortable readout: honest interval $20 .. $69 around $31 — note the longer expensive arm (bounded)


So what *should* go on the planning slide? Two numbers, both built from the
detectable-lift **bound** rather than any flipped estimate:

- the design's **maximum detectable cost per conversion** — the worst CPA the
  test can still tell apart from "the media did nothing", and
- the **power to certify a CPA target** — the chance the test comes back with
  its (asymmetric) upper bound below the number the client cares about. Because
  the expensive arm of the interval is long, *certifying* a CPA is much harder
  than merely estimating one — the curve below is the honest version of that
  conversation.

In [22]:
from mmm_framework.planning import cpa_power
from mmm_framework.planning.methods import ghost_ads_users_for_cpa

print(f"this design's MAX detectable CPA: ${p['max_detectable_cpa']:.0f}  "
      f"(= cost/user ${COST_PER_USER:.02f} ÷ MDE lift {p['mde_abs']:.4f}, {p['cpa_basis']} basis)")
for target in (20.0, 35.0):
    n = ghost_ads_users_for_cpa(design, target)
    print(f"users so that a true CPA of ${target:.0f} is detectable: {n:,}")
print(f"\npower to come away with a BOUNDED CPA at all: "
      f"{cpa_power(COST_PER_USER, p['se_null'], true_lift):.0%}")
for target in (40.0, 60.0, 100.0):
    pw = cpa_power(COST_PER_USER, p["se_null"], true_lift, target_cpa=target)
    print(f"power to CERTIFY 'CPA ≤ ${target:.0f}' (truth ${tc:.0f}): {pw:.0%}")

reach = np.array([50_000, 100_000, 200_000, 400_000, 800_000, 1_600_000, 3_200_000])
cap = [ghost_ads_power(GhostAdsDesign(users_reached=int(n), baseline_rate=0.021,
                                      exposure_rate=0.65, cost_per_user=0.05)
                       )["max_detectable_cpa"] for n in reach]
targets = np.linspace(tc * 1.05, tc * 5, 60)
certify = [cpa_power(COST_PER_USER, p["se_null"], true_lift, target_cpa=float(t))
           for t in targets]

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.13, subplot_titles=(
    "Reach buys a worse-CPA detection ceiling",
    f"Power to certify 'CPA ≤ target' (truth ${tc:.0f})"))
fig.add_trace(go.Scatter(x=reach, y=cap, mode="lines+markers",
    line=dict(color=PALETTE["Search"], width=3), showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=[design.users_reached], y=[p["max_detectable_cpa"]],
    mode="markers", marker=dict(size=15, symbol="star", color=GOLD,
    line=dict(width=1.5, color=INK)), showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=targets, y=certify, mode="lines",
    line=dict(color=GOOD, width=3), showlegend=False), row=1, col=2)
fig.add_hline(y=0.8, line=dict(color=INK, width=1.5, dash="dot"),
              annotation_text="80% bar", row=1, col=2)
fig.add_vline(x=tc, line=dict(color=INK, width=1.5, dash="dash"),
              annotation_text="true CPA", row=1, col=2)
fig.update_xaxes(title_text="users reached", type="log", row=1, col=1)
fig.update_yaxes(title_text="max detectable CPA ($)", row=1, col=1)
fig.update_xaxes(title_text="CPA target to certify ($)", row=1, col=2)
fig.update_yaxes(title_text="power", tickformat=".0%", row=1, col=2)
style(fig, height=380,
      title="Planning on the CPA scale — built from the detectable-lift bound, never the flipped estimate")
fig.show()
print("\nNote how far right of the TRUE CPA the certification curve crosses 80%:")
print("the long expensive arm of the interval is what the client is really paying to shorten.")

this design's MAX detectable CPA: $39  (= cost/user $0.05 ÷ MDE lift 0.0013, itt basis)
users so that a true CPA of $20 is detectable: 109,274
users so that a true CPA of $35 is detectable: 326,787

power to come away with a BOUNDED CPA at all: 94%
power to CERTIFY 'CPA ≤ $40' (truth $31): 12%
power to CERTIFY 'CPA ≤ $60' (truth $31): 40%
power to CERTIFY 'CPA ≤ $100' (truth $31): 69%



Note how far right of the TRUE CPA the certification curve crosses 80%:
the long expensive arm of the interval is what the client is really paying to shorten.


## 5 · Switchbacks — time-randomized on/off blocks

No geo split, no user-level randomization? Randomize **time**. Two things make
naive switchback power dishonest, and the design handles both:

- **Carryover** — a block shorter than the adstock memory smears treatment into
  control blocks; the block length respects the washout and the analysis plan
  drops a burn-in at each boundary.
- **Autocorrelation** — serially-correlated weeks mean fewer effective
  observations; the design effect uses the AR(1) coefficient of the *detrended
  levels* (first-differencing a smooth series flips the correlation negative
  and silently zeroes the correction).

In [23]:
from mmm_framework.planning.methods import switchback_design

sb = switchback_design(NAT_CSV, KPI, "TV", duration=12, amplitude_pct=50.0)
pw = sb["switchback_power"]
ar1 = sb["ar1"]
print(f"block length: {sb['block_weeks']}w  (carryover {sb['carryover_weeks']}w, "
      f"{sb['carryover_source']})   burn-in {sb['burn_in_weeks']}w/block   "
      f"{sb['n_blocks']} blocks, {sb['n_switches']} switches")
print(f"AR(1) rho (detrended levels): {ar1['rho']:.2f}  ->  design effect {ar1['deff']:.2f}")
print(f"SE iid: {pw['se_iid']:.1f}   SE honest: {pw['se_honest']:.1f}   "
      f"MDE honest: {pw['mde_honest']:.1f} {pw['mde_units']}")

sched = pd.DataFrame(sb["schedule"])
fig = make_subplots(rows=1, cols=2, column_widths=[0.62, 0.38], horizontal_spacing=0.14,
                    subplot_titles=("Randomized block schedule", "The honesty tax"))
fig.add_trace(go.Bar(x=sched["week_offset"], y=sched["multiplier"],
                     marker_color=PALETTE["TV"], showlegend=False), row=1, col=1)
fig.add_hline(y=1.0, line=dict(color=INK, width=1, dash="dot"), row=1, col=1)
fig.add_trace(go.Bar(x=["iid<br>(naive)", "AR(1)-honest"], y=[pw["mde_iid"], pw["mde_honest"]],
                     marker_color=[MUTED, PALETTE["TV"]],
                     text=[f"{pw['mde_iid']:.0f}", f"{pw['mde_honest']:.0f}"],
                     textposition="outside", showlegend=False), row=1, col=2)
fig.update_xaxes(title_text="week", row=1, col=1)
fig.update_yaxes(title_text="spend x BAU", row=1, col=1)
fig.update_yaxes(title_text="MDE (KPI/week)", row=1, col=2)
style(fig, height=360, title="Switchback: carryover-aware blocks + the AR(1) design effect")
fig.show()
assert pw["se_honest"] >= pw["se_iid"]   # autocorrelation can only cost you

block length: 2w  (carryover 2w, assumed)   burn-in 1w/block   6 blocks, 5 switches
AR(1) rho (detrended levels): 0.87  ->  design effect 1.87
SE iid: 117.4   SE honest: 160.7   MDE honest: 449.9 KPI per week (on-vs-off step)


## 6 · The methodology leaderboard — audition estimators on your own history

Analytic power assumes away the mess in real panels. The leaderboard runs every
estimator the data supports through **A/A** (no effect — does its 5% test
really reject 5%?) and **A/B** (inject a known lift — what does it actually
detect?) *on your own history*, then ranks by **validity → power → cost**. An
estimator whose naive test fires on noise gets a block-calibrated critical
value before it is allowed to compete.

In [24]:
from mmm_framework.planning import methodology_leaderboard

board = methodology_leaderboard(GEO_CSV, KPI, "TV", duration=8,
                                max_aa_windows=40, max_ab_windows=24, seed=42)
lb = pd.DataFrame(board["methodologies"])[
    ["key", "valid", "fpr", "fpr_at_crit", "empirical_mde_roas", "power_at_expected_effect"]
]
print(lb.round(3).to_string(index=False))
print(f"\nrecommended method on this history: {board['chosen_key']}")

fig = go.Figure()
fig.add_trace(go.Bar(name="naive 5% rule", x=lb["key"], y=lb["fpr"], marker_color=BAD))
fig.add_trace(go.Bar(name="block-calibrated", x=lb["key"], y=lb["fpr_at_crit"], marker_color=GOOD))
fig.add_hline(y=0.05, line=dict(color=INK, width=1.5, dash="dot"),
              annotation_text="honest 5% size")
style(fig, height=380, title="A/A false-positive rate by estimator — validity before power",
      barmode="group", legend=dict(orientation="h", y=-0.25))
fig.update_layout(yaxis_tickformat=".0%", yaxis_title="A/A false-positive rate")
fig.show()

              key  valid   fpr  fpr_at_crit empirical_mde_roas  power_at_expected_effect
       pooled_did   True 0.150         0.05               None                     0.958
     per_pair_did   True 0.125         0.05               None                     0.958
synthetic_control   True   NaN         0.05               None                     0.625
              tbr   True 0.125         0.05               None                     0.875
              gbr   True 0.125         0.05               None                     0.958
       regadj_geo  False 0.275         0.05               None                     0.875

recommended method on this history: pooled_did


## 7 · Fit the MMM — the model that anchors everything downstream

Everything so far ran **pre-fit** (pure pandas + numpy). The rest of the
playbook — expected effects, EIG/EVOI, economics, the net-value optimizer —
is anchored on a fitted model. A short NUTS run is enough for a demo.

In [25]:
from mmm_framework.agents.fitting import build_and_fit

spec = {
    "kpi": KPI, "kpi_level": "geo",
    "media_channels": [{"name": n} for n in CHANNELS],
    "control_variables": [],
    "inference": {"draws": 150, "tune": 150, "chains": 2, "random_seed": 0},
    "seasonality": {"yearly": 2},
    "trend": {"type": "linear"},
}
model, results, _ = build_and_fit(spec, GEO_CSV)
print(f"fitted: {len(model.channel_names)} channels, {model.n_geos} geos, "
      f"{model.n_periods} periods  (approximate={results.approximate})")

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


Output()

Output()

...

Output()

Output()

Output()

Output()

fitted: 4 channels, 8 geos, 78 periods  (approximate=False)


## 8 · Which channel deserves a test? — the EIG/EVOI priority grid

Two axes separate "wide posterior" from "worth testing":

- **EIG** — what a test of achievable precision would *teach*:
  $\mathrm{EIG} = \tfrac12 \ln(1 + \sigma_k^2/\sigma_{\text{exp}}^2)$ nats.
- **EVOI** — what that learning is *worth to the budget decision*
  (preposterior Monte-Carlo: simulate outcomes, reweight, re-optimize), capped
  at EVPI.

The quadrants name the verdicts: `test_now` (uncertain **and** budget-critical),
`learn_cheaply`, `monitor`, `deprioritize`. A channel can have huge EIG and
near-zero EVOI — you'd learn a lot, but the budget wouldn't move.

In [26]:
from mmm_framework.planning import compute_experiment_priorities

grid, portfolio = compute_experiment_priorities(model, max_draws=120, random_seed=42)
gdf = pd.DataFrame([g.to_dict() for g in grid])[
    ["channel", "spend_share", "roi_mean", "roi_sd", "sigma_exp", "eig", "evoi", "quadrant", "priority"]
]
print(gdf.round(3).to_string(index=False))
print(f"\nportfolio EVPI (value of PERFECT information): {portfolio['evpi']:,.1f} KPI-units")

fig = go.Figure()
for _, r in gdf.iterrows():
    fig.add_trace(go.Scatter(
        x=[r["eig"]], y=[r["evoi"]], mode="markers+text", text=[r["channel"]],
        textposition="top center", name=r["channel"],
        marker=dict(size=14 + 60 * r["spend_share"], color=PALETTE[r["channel"]],
                    line=dict(color="white", width=1.5)), showlegend=False))
fig.add_vline(x=portfolio["eig_threshold"], line=dict(color=MUTED, width=1, dash="dot"))
fig.add_hline(y=portfolio["evoi_threshold"], line=dict(color=MUTED, width=1, dash="dot"))
style(fig, height=420, title="EIG x EVOI — top-right (test_now) earns the experiment; bubble = spend share")
fig.update_layout(xaxis_title="EIG (nats) — what the test teaches",
                  yaxis_title="EVOI (KPI units) — what the learning is worth")
fig.show()
top = gdf.iloc[0]["channel"]
print(f"top priority: {top} ({gdf.iloc[0]['quadrant']})")

channel  spend_share  roi_mean  roi_sd  sigma_exp   eig    evoi      quadrant  priority
     TV        0.389     0.312   0.159      0.030 1.738 494.481       monitor     0.938
 Search        0.249     0.471   0.242      0.043 1.777 331.039       monitor     0.776
 Social        0.205     0.374   0.218      0.031 1.976 257.241 learn_cheaply     0.721
Display        0.157     0.429   0.265      0.038 1.976 235.281 learn_cheaply     0.690

portfolio EVPI (value of PERFECT information): 1,215.8 KPI-units


top priority: TV (monitor)


## 9 · Non-geo designs compete on the same scale — the `sigma_exp` bridge

Nothing in EIG/EVOI is geographic. Both consume exactly one fact about a
candidate design: the standard error it would achieve **on the ROI scale**.
Every planner above emits one:

| design | its $\sigma_{\text{exp}}$ |
|---|---|
| geo holdout / scaling | power-curve MDE at the chosen duration ÷ 2.8 |
| ghost ads | lift MDE × value per conversion ÷ test spend ÷ 2.8 |
| switchback | `switchback_power`'s honest SE, bridged by weekly spend |
| flighting | the design-matrix `se_roas` / `se_mroas` |

So a user-level RCT on Search competes against a geo holdout on TV *on one
scale* — the experiment-selection problem stops caring what kind of experiment
it is.

In [27]:
# Bridge the ghost-ads design to the ROI scale for Search:
# MDE in conversions x $/conversion / media cost -> an ROAS-scale MDE -> /2.8 -> sigma_exp.
ghost_roas_mde = p["incremental_value_at_mde"] / p["media_cost"]
ghost_sigma = ghost_roas_mde / 2.8
print(f"ghost-ads bridge for Search: ROAS-scale MDE {ghost_roas_mde:.2f} -> sigma_exp {ghost_sigma:.3f}")

grid2, _ = compute_experiment_priorities(
    model, max_draws=120, random_seed=42,
    sigma_exp_overrides={"Search": ghost_sigma},
)
g2 = pd.DataFrame([g.to_dict() for g in grid2])[["channel", "sigma_exp", "eig", "evoi", "quadrant"]]
before = gdf.set_index("channel").loc["Search"]
after = g2.set_index("channel").loc["Search"]
print(g2.round(3).to_string(index=False))
print(f"\nSearch under its own achievable design: sigma_exp {before['sigma_exp']:.3f} -> "
      f"{after['sigma_exp']:.3f},  EIG {before['eig']:.2f} -> {after['eig']:.2f} nats")

ghost-ads bridge for Search: ROAS-scale MDE 0.98 -> sigma_exp 0.350
channel  sigma_exp   eig    evoi      quadrant
     TV      0.030 1.738 494.481       monitor
 Social      0.031 1.976 257.241      test_now
Display      0.038 1.976 235.281 learn_cheaply
 Search      0.350 0.291  23.103  deprioritize

Search under its own achievable design: sigma_exp 0.043 -> 0.350,  EIG 1.78 -> 0.29 nats


## 10 · What does the test cost, and what is it worth? — net test value

EVOI prices the upside; a real test also has a downside — the profit given up
while it runs. `compute_experiment_net_value` nets the two into one figure:

$$\text{net value} = \underbrace{\min(\mathrm{EVOI}, \mathrm{EVPI}) \cdot m \cdot \bar d}_{\text{decayed reallocation gain}} + \underbrace{\mathbb{E}[m\,\Delta\mathrm{KPI} - \Delta\mathrm{spend}]}_{\text{signed profit during the test}}$$

with a **decay haircut** $\bar d$ (information has a half-life; EVOI values a
posterior that stays sharp forever) and an **EVPI cap**. The signed convention
keeps a money-saving holdout coherent: saved spend can outweigh forgone margin,
making the "loss" negative and the break-even horizon zero.

In [28]:
from mmm_framework.planning import compute_experiment_net_value, compute_opportunity_cost
from mmm_framework.planning.eig import channel_half_life

MARGIN = 0.5
row = gdf.set_index("channel").loc[top]
test_design = design_experiment(GEO_CSV, KPI, top, design="holdout", n_pairs=4, duration=8)

oc = compute_opportunity_cost(model, test_design, margin_per_kpi=MARGIN,
                              evoi_kpi_units=float(row["evoi"]),
                              max_draws=120, return_draws=True)
nv = compute_experiment_net_value(
    channel=top,
    evoi_kpi_units=float(row["evoi"]),
    evpi_kpi_units=float(portfolio["evpi"]),
    margin_per_kpi=MARGIN,
    response_horizon_weeks=26,
    half_life_weeks=channel_half_life(top),
    model_anchored=True,
    opportunity_cost_result=oc,
)
print(f"reallocation gain (decayed, capped): ${nv.reallocation_gain:,.0f}   "
      f"(decay factor {nv.decay_factor:.2f}, half-life {nv.half_life_weeks:.0f}w)")
print(f"signed profit during the test:       ${nv.net_profit_during_test:,.0f}")
print(f"NET VALUE of running this test:      ${nv.net_value:,.0f}")
print(f"P(net > 0): {nv.prob_net_positive:.0%}    "
      f"break-even horizon: {nv.breakeven_horizon_weeks} weeks    basis: {nv.basis}")

parts = [("gain from the learning", nv.reallocation_gain),
         ("profit impact during test", nv.net_profit_during_test),
         ("net value", nv.net_value)]
fig = go.Figure(go.Bar(x=[a for a, _ in parts], y=[b for _, b in parts],
                       marker_color=[GOOD if v >= 0 else BAD for _, v in parts],
                       text=[f"${v:,.0f}" for _, v in parts], textposition="outside",
                       showlegend=False))
fig.add_hline(y=0, line=dict(color=INK, width=1))
style(fig, height=360, title=f"Is the {top} test worth running? gain vs loss, in dollars")
fig.show()

reallocation gain (decayed, capped): $210   (decay factor 0.85, half-life 52w)
signed profit during the test:       $1,862
NET VALUE of running this test:      $2,072
P(net > 0): 100%    break-even horizon: 0.0 weeks    basis: model_anchored


## 11 · **New** — the Pareto optimizer prices every design in net dollars

Within one channel there is still a design space: holdout vs scaling,
footprint, intensity, duration. `suggest_experiment` sweeps a bounded grid on
four lower-is-better objectives — MDE, power shortfall, cost, duration — and
returns the non-dominated **Pareto front** plus the powered "knee".

**What's new (2026-07-19):** with a margin, the cost axis is upgraded to the
**net value of testing** per candidate. The expensive part — a preposterior
Monte-Carlo EVOI per design — is priced by a **calibrated Gaussian surrogate**
(Raiffa–Schlaifer preposterior form):

$$\mathrm{EVOI}(\sigma) \approx k \cdot s(\sigma)\,\Psi\!\big(\delta/s(\sigma)\big),
\qquad s(\sigma) = \tau\sqrt{\tau^2/(\tau^2+\sigma^2)}$$

with $(k, \delta)$ fitted to **two** anchored MC EVOIs placed at the extremes
of the grid's design precisions — every candidate interpolates inside the
calibrated bracket. The front then reads the way a CFO wants it read: designs
that are precise, powered, fast, *and expected to create money net of their own
cost*.

In [29]:
from mmm_framework.planning import suggest_experiment

out = suggest_experiment(
    model, GEO_CSV, KPI, top,
    margin=MARGIN,
    duration_min=6, duration_max=16,
    intensity_min=-100, intensity_max=100,
    max_draws=60, random_seed=42,
)
assert out["net_value_axis"] is True, "margin known -> the net-value axis engages"

anch = out["evoi_anchor"]
print(f"EVOI surrogate: {len(anch['anchors'])} MC anchors at sigma = "
      f"{[round(a[0], 3) for a in anch['anchors']]}  ->  k={anch['k']:.1f}, "
      f"delta={anch['delta']:.4f}, tau={anch['tau']:.4f}, EVPI={anch['evpi']:.1f}")
print(f"cost axis: {out['tradeoff_label']}\n")

cands = pd.DataFrame(out["candidates"])
front = cands[cands["on_pareto"]]
cols = ["mode", "footprint", "intensity_pct", "duration", "mde_roas", "power",
        "evoi_kpi", "reallocation_gain", "net_value", "powered", "is_recommended"]
print("the Pareto front (non-dominated designs), priced in net dollars:")
print(front[cols].round(2).to_string(index=False))

rec = out["recommended"]
dom = cands[~cands["on_pareto"] & ~cands["is_recommended"]]
fr = cands[cands["on_pareto"] & ~cands["is_recommended"]]
fig = go.Figure()
fig.add_trace(go.Scatter(x=dom["mde_roas"], y=dom["net_value"], mode="markers",
    name="dominated", marker=dict(size=9, color=MUTED, opacity=0.45)))
fig.add_trace(go.Scatter(x=fr["mde_roas"], y=fr["net_value"], mode="markers",
    name="Pareto front", marker=dict(size=14, color=fr["duration"], colorscale="YlGnBu",
    reversescale=True, colorbar=dict(title="weeks", thickness=10, len=0.6),
    line=dict(width=2, color=INK))))
fig.add_trace(go.Scatter(x=[rec["mde_roas"]], y=[rec["net_value"]], mode="markers",
    name="recommended", marker=dict(size=20, symbol="star", color=GOLD,
    line=dict(width=1.5, color=INK))))
fig.add_hline(y=0, line=dict(color=INK, width=1, dash="dot"),
              annotation_text="test pays for itself above this line")
style(fig, height=430,
      title=f"{top}: the design space priced in dollars — upper-left is best (precise AND net-positive)",
      legend=dict(orientation="h", y=-0.22))
fig.update_layout(xaxis_title="MDE (ROAS) — lower is more precise",
                  yaxis_title="net value of testing ($)")
fig.show()
print(f"\nrecommended: {rec['mode']} / {rec['footprint']} / {rec['intensity_pct']:+.0f}% / "
      f"{rec['duration']}w — net value ${rec['net_value']:,.0f}, "
      f"cool-down {out['cooldown']['cooldown_weeks']}w before a clean re-read")

EVOI surrogate: 2 MC anchors at sigma = [0.053, 0.261]  ->  k=16534.2, delta=0.0059, tau=0.0657, EVPI=1028.8
cost axis: net value of testing ($, higher is better)

the Pareto front (non-dominated designs), priced in net dollars:
   mode footprint  intensity_pct  duration  mde_roas  power  evoi_kpi  reallocation_gain  net_value  powered  is_recommended
holdout      full         -100.0         6      0.23   0.21    224.28              95.40    1470.12    False           False
holdout      full         -100.0        11      0.18   0.29    267.39             113.74    2582.97    False            True
holdout      full         -100.0        16      0.15   0.35    289.94             123.33    3724.77    False           False



recommended: holdout / full / -100% / 11w — net value $2,583, cool-down 6w before a clean re-read


## 12 · Trust but verify — the surrogate against the exact preposterior MC

The net-value axis leans on the surrogate, so check it against ground truth:
run the *exact* preposterior Monte-Carlo EVOI at several design precisions and
overlay the fitted curve. Anchored at the extremes, every candidate is an
**interpolation** — the regime where the surrogate stays within a few tens of
percent of the MC, which is what the Pareto axis needs: it has to *rank*
designs correctly, not reproduce each EVOI to the decimal. (Extrapolating far
beyond the weak anchor under-estimates — which is exactly why the optimizer
anchors at the extremes of its own grid.)

In [30]:
from mmm_framework.planning import (
    compute_evoi_for_channel, compute_evpi, compute_response_curves,
    fit_evoi_surrogate, optimize_budget,
)

curves = compute_response_curves(model, max_draws=60, random_seed=42)
ci = curves.channel_names.index(top)
g1 = int(np.argmin(np.abs(curves.multipliers - 1.0)))
roi_draws = curves.contributions[:, ci, g1] / max(float(curves.base_spend[ci]), 1e-12)
tau = float(roi_draws.std())

opt = optimize_budget(curves=curves, random_seed=42)
port = compute_evpi(curves, total_budget=opt.total_budget,
                    per_draw_alloc=opt.per_draw_alloc, optimal_alloc=opt.optimal_alloc)

rng = np.random.default_rng(42)
od = (rng.integers(0, len(roi_draws), size=48), rng.standard_normal(48))
sig_lo, sig_hi = tau / 2, 3 * tau
anchors = [(s, compute_evoi_for_channel(curves, ci, roi_draws, float(s),
                                        optimal_alloc=opt.optimal_alloc,
                                        total_budget=opt.total_budget,
                                        outcome_draws=od))
           for s in (sig_lo, sig_hi)]
sur = fit_evoi_surrogate(tau, anchors)

sig_grid = np.linspace(sig_lo, sig_hi, 9)
mc = [compute_evoi_for_channel(curves, ci, roi_draws, float(s),
                               optimal_alloc=opt.optimal_alloc,
                               total_budget=opt.total_budget, outcome_draws=od)
      for s in sig_grid]
sg = np.linspace(sig_lo * 0.8, sig_hi * 1.1, 80)
# NB: the raw (uncapped) surrogate vs the raw MC — the fidelity question.
# In production the optimizer additionally caps at EVPI (a conservative floor
# that can clip everything to zero when the portfolio decision is already firm).
fig = go.Figure()
fig.add_trace(go.Scatter(x=sg, y=[sur(float(s)) for s in sg],
    mode="lines", name="Gaussian surrogate (2 anchors)",
    line=dict(color=PALETTE[top], width=3)))
fig.add_trace(go.Scatter(x=sig_grid, y=mc, mode="markers", name="exact preposterior MC",
    marker=dict(size=10, color=INK, symbol="circle-open", line=dict(width=2))))
fig.add_trace(go.Scatter(x=[a[0] for a in anchors], y=[a[1] for a in anchors],
    mode="markers", name="MC anchors", marker=dict(size=14, color=GOLD,
    line=dict(width=1.5, color=INK))))
style(fig, height=400,
      title=f"{top}: surrogate vs exact EVOI — two MC evaluations price the whole design grid",
      legend=dict(orientation="h", y=-0.22))
fig.update_layout(xaxis_title="design precision sigma_exp (ROI scale)",
                  yaxis_title="EVOI (KPI units)")
fig.show()

interior = [(s, m, sur(float(s))) for s, m in zip(sig_grid, mc) if m > 0]
ratios = [su / m for _, m, su in interior]
print(f"surrogate/MC ratio across the bracket: "
      f"min {min(ratios):.2f}, median {np.median(ratios):.2f}, max {max(ratios):.2f}")
print(f"portfolio EVPI (the production cap on any gain): {port.evpi:,.1f} KPI-units")

surrogate/MC ratio across the bracket: min 1.00, median 1.27, max 1.42
portfolio EVPI (the production cap on any gain): 1,028.8 KPI-units


## 13 · Close the loop — the readout calibrates the next fit

The test ran (here: simulated). Its result is direct evidence about the tested
channel that the next fit must not ignore — folded in as a **likelihood term on
the model's own estimand**: the measured incremental ROAS, with its standard
error, constrains the in-graph ROAS of that channel over that window.

Two outcomes are possible, and both are informative. If the readout **agrees**
with the observational fit, the posterior sharpens where the test measured. If
it **disagrees** — as the simulated readout below deliberately does — the
posterior is *pulled toward the measured value*, and the residual tension is a
**model-misspecification signal** that surfaces in diagnostics instead of being
silently overwritten. That visibility is the point of the likelihood route over
hand-tightening a prior: a hand-set prior would have hidden the conflict.

In [31]:
from mmm_framework.calibration import ExperimentEstimand, ExperimentMeasurement

# ROI posterior BEFORE calibration (per-draw ROI at current spend)
spend_top = float(model.X_media_raw[:, ci].sum())
contrib_pre = model.sample_channel_contributions(max_draws=200, random_seed=7)
roi_pre = contrib_pre[:, :, ci].sum(axis=1) / spend_top

# the (simulated) readout: the holdout measured incremental ROAS = 1.2 +/- 0.25
# — deliberately HIGHER than the observational fit believes, so we can watch
# the likelihood pull the posterior and expose the tension.
exp = ExperimentMeasurement(
    channel=top,
    test_period=(60, 67),                 # integer period indices into the panel
    value=1.2, se=0.25,
    estimand=ExperimentEstimand.ROAS,
)
model.add_experiment_calibration([exp])
results_cal = model.fit(draws=150, tune=150, chains=2, random_seed=0)

contrib_post = model.sample_channel_contributions(max_draws=200, random_seed=7)
roi_post = contrib_post[:, :, ci].sum(axis=1) / spend_top

fig = go.Figure()
for name, draws, color in [("before calibration", roi_pre, MUTED),
                           ("after calibration", roi_post, PALETTE[top])]:
    fig.add_trace(go.Histogram(x=draws, name=name, opacity=0.6, nbinsx=40,
                               marker_color=color, histnorm="probability density"))
fig.add_vline(x=1.2, line=dict(color=INK, width=2, dash="dash"),
              annotation_text="experiment readout (1.2)")
style(fig, height=380, barmode="overlay",
      title=f"{top} ROI posterior: the readout's likelihood pulls it toward the measured value",
      legend=dict(orientation="h", y=-0.22))
fig.update_layout(xaxis_title="ROI (window total)", yaxis_title="density")
fig.show()
shift_se = abs(roi_post.mean() - roi_pre.mean()) / max(roi_pre.std(), 1e-9)
print(f"ROI posterior mean: {roi_pre.mean():.2f} -> {roi_post.mean():.2f} "
      f"(readout said 1.2 +/- 0.25) — a {shift_se:.1f}-sd shift toward the experiment")
print(f"ROI posterior sd:   {roi_pre.std():.3f} -> {roi_post.std():.3f}")
print("The posterior lands BETWEEN the observational fit and the readout, and the")
print("unresolved gap is a misspecification flag to chase (missing confounder,")
print("wrong saturation), not a number to overwrite.")

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


ROI posterior mean: 0.31 -> 0.75 (readout said 1.2 +/- 0.25) — a 2.8-sd shift toward the experiment
ROI posterior sd:   0.158 -> 0.195
The posterior lands BETWEEN the observational fit and the readout, and the
unresolved gap is a misspecification flag to chase (missing confounder,
wrong saturation), not a number to overwrite.


## Wrap-up — the playbook in one breath

1. **Enumerate** what your data supports (`list_methods` / `methods_for_data` /
   `design_options`) — geo panels unlock SCM/TBR/GBR/DiD; no panel still leaves
   ghost ads, switchbacks and flighting.
2. **Plan** with honest power math — matched pairs and power curves for geo,
   ITT dilution and rare-event flags for ghost ads, carryover-aware blocks and
   AR(1) design effects for switchbacks.
3. **Translate power to the client's metric** — the **maximum detectable cost
   per conversion** (`planning.cpa`): invert the detectable-lift *bound*, never
   the point estimate — the CPA readout is right-skewed, its mean overshoots,
   the symmetric ± flip under-covers (and can go negative), and a weak readout's
   honest interval is *unbounded above*.
4. **Audition the estimator** on your own history (`methodology_leaderboard`) —
   validity (A/A) before power (A/B).
5. **Pick the channel** with EIG × EVOI — and bring *non-geo* designs onto the
   same scale through the `sigma_exp` bridge.
6. **Price the test** — opportunity cost, decayed/EVPI-capped gain, one net
   dollar figure with `P(net > 0)` and a break-even horizon.
7. **Optimize the design** — the Pareto front now ranks candidates by the **net
   value of testing**, priced by the two-anchor Gaussian EVOI surrogate.
8. **Calibrate** — the readout becomes a likelihood term; the posterior moves
   toward what the test measured.

**Further reading:** [`docs/experiment-playbook.html`](../../docs/experiment-playbook.html)
(the reference version of this playbook), the `lifecycle_00–06` notebook series
(the same loop run as an ongoing measurement program), and
`technical-docs/experiment-net-economics.md` (the net-value spec, including the
surrogate's derivation and its verified accuracy envelope).